In [1]:
#Library yang dibutuhkan (pastikan untuk install selenium dahulu)
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.support.ui import WebDriverWait
import time, pandas as pd

In [1]:
pip install selenium


Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Pengaturan agar menerima review dengan Bahasa Indonesia
options = Options()
options.add_argument('--lang=id')
options.add_argument('accept-langguange=id-ID, id')
driver = webdriver.Chrome(options=options)

# Panggil lokasi digmaps lewat URL (Pastikan link yang dicopy pada halaman review)
driver.get("https://www.google.com/maps/place/Pantai+Labuhan+Jukung/@-5.1901565,103.9213373,15z/data=!4m12!1m2!2m1!1spantai+labuhan+jukung!3m8!1s0x2e4794d855860b85:0xb5dce6222d97ea34!8m2!3d-5.1909103!4d103.9310314!9m1!1b1!15sChVwYW50YWkgbGFidWhhbiBqdWt1bmdaFyIVcGFudGFpIGxhYnVoYW4ganVrdW5nkgESdG91cmlzdF9hdHRyYWN0aW9umgFEQ2k5RFFVbFJRVU52WkVOb2RIbGpSamx2VDJ0a2IxWnVhekpVVjBwNFRteFNWbUpITVVaaVZURjNWMFZzUm1WSFl4QULgAQD6AQQIABBD!16s%2Fg%2F11dxl7tyyh?entry=ttu&g_ep=EgoyMDI2MDIyMy4wIKXMDSoASAFQAw%3D%3D")
time.sleep(3)

# Cek apakah lokasi benar-benar ada
try:
    nama_tempat = driver.find_element(By.CLASS_NAME, "DUwDvf").text.strip()
except NoSuchElementException:
    nama_tempat = "unknown"

# Klik tombol More Reviews jika ada (untuk mendapatkan ulasan dengan lengkap)
try:
    more_reviews = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, "//span[contains(text(), 'Ulasan lainnya') or contains(text(), 'More reviews')]/ancestor::button")))
    more_reviews.click()
    print("[INFO] Klik tombol More reviews.")
    time.sleep(3)
except:
    print("[INFO] Tidak ada tombol More reviews.")

# Scraping data di Gmaps
reviews, seen = [], set()
while len(reviews) < 50:
    #Tentukan Element yang ingin diambil pada halaman inspect GMAPS)
    for c in driver.find_elements(By.CSS_SELECTOR, 'div.jftiEf'): # Container seluruh Review
        user_el = c.find_elements(By.CLASS_NAME, 'd4r55') # Element Nama User
        review_el = c.find_elements(By.CLASS_NAME, 'wiI7pd') # Element Isi Review
        rating_el = c.find_elements(By.CLASS_NAME, 'kvMYJc') # Element Rating (Bintang)
        
        # Skip jika review kosong
        if not review_el or not review_el[0].text.strip():
            continue
        user = user_el[0].text if user_el else 'Unknown'
        review = review_el[0].text.strip()
        rating = rating_el[0].get_attribute('aria-label').split()[0] if rating_el else 'Unknown'
        if (user, review) not in seen:
            seen.add((user, review))
            reviews.append({
                'nama_tempat': nama_tempat,
                'user': user,
                'review': review,
                'rating': rating
            })
    try:
        driver.execute_script(
            "arguments[0].scrollIntoView();",
            driver.find_elements(By.CSS_SELECTOR, 'div.jftiEf')[-1]
        )
    except:
        break
    time.sleep(2)


[INFO] Tidak ada tombol More reviews.


In [3]:
# Menyimpan hasil dari scraping GMAPS
pd.DataFrame(reviews).to_csv("ulasan_rsuad.csv", index=False)
print(f"[INFO] {len(reviews)} ulasan berhasil didapatkan dan di eksport")
driver.quit


[INFO] 58 ulasan berhasil didapatkan dan di eksport


<bound method ChromiumDriver.quit of <selenium.webdriver.chrome.webdriver.WebDriver (session="689b372b4e9dd83fe4565500545b199e")>>